# Fine-Tune InternVL2-4B on Cantopop Lyrics Corpus

This notebook fine-tunes **OpenGVLab/InternVL2-4B** using **LoRA (QLoRA)** on the `cantopop_corpus_final_583_yue.csv` dataset for Cantonese lyrics generation.

**Target task:** Given a scene/image description, generate structured Cantonese lyrics in YuE format (`[verse]` / `[chorus]`).

> ⚠️ Run this on **Google Colab with a T4/A100 GPU**. Go to `Runtime → Change runtime type → GPU`.


## 1. Install Required Libraries

In [ ]:
!pip install -q transformers==4.40.0 datasets peft trl accelerate bitsandbytes sentencepiece einops torchvision
!pip install -q flash-attn --no-build-isolation 2>/dev/null || echo "flash-attn not available; continuing without it"

## 2. Mount Google Drive & Load CSV

Upload `cantopop_corpus_final_583_yue.csv` to your Google Drive (e.g. `MyDrive/cantopop/`), then mount below.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import pandas as pd

# ── Update this path to match where you uploaded the CSV in your Drive ──
CSV_PATH = "/content/drive/MyDrive/cantopop/cantopop_corpus_final_583_yue.csv"

df = pd.read_csv(CSV_PATH)
print(f"Loaded {len(df)} rows")
print(df.columns.tolist())
df.head(3)

## 3. Preprocess & Tokenize Dataset

We format each row as a **prompt → response** pair:
- **Prompt:** Artist/lyricist/year metadata + instruction to generate Cantonese lyrics
- **Response:** The `Lyrics_YuE` column (already structured with `[verse]`/`[chorus]`)

In [ ]:
import re

MODEL_ID = "OpenGVLab/InternVL2-4B"
MAX_LENGTH = 1024  # adjust if you hit OOM

def build_instruction(row):
    """Build the instruction prompt from CSV metadata."""
    artist   = str(row.get("Artist", "")).strip()
    title    = str(row.get("Title", "")).strip()
    lyricist = str(row.get("Lyricist", "")).strip()
    year     = str(row.get("Released on", "")).strip()[:4]  # just the year
    return (
        f"你是一個粵語流行歌詞創作助手。\n"
        f"請根據以下資訊，創作一首粵語歌詞，分為 [verse] 和 [chorus]。\n\n"
        f"歌手：{artist}\n"
        f"歌名：{title}\n"
        f"作詞：{lyricist}\n"
        f"年份：{year}\n\n"
        f"請直接輸出歌詞，不要有任何額外說明。"
    )

def build_sample(row):
    instruction = build_instruction(row)
    lyrics = str(row.get("Lyrics_YuE", "")).strip()
    # InternVL2 chat template format
    return f"<|im_start|>user\n{instruction}<|im_end|>\n<|im_start|>assistant\n{lyrics}<|im_end|>"

# Drop rows with empty lyrics
df = df.dropna(subset=["Lyrics_YuE"])
df = df[df["Lyrics_YuE"].str.strip().str.len() > 50].reset_index(drop=True)
print(f"Clean samples: {len(df)}")

samples = [build_sample(row) for _, row in df.iterrows()]
print("Example sample:\n")
print(samples[0][:600])

In [ ]:
from transformers import AutoTokenizer
from datasets import Dataset

tokenizer = AutoTokenizer.from_pretrained(MODEL_ID, trust_remote_code=True, use_fast=False)
tokenizer.pad_token = tokenizer.eos_token

def tokenize(sample):
    out = tokenizer(
        sample["text"],
        truncation=True,
        max_length=MAX_LENGTH,
        padding="max_length",
    )
    out["labels"] = out["input_ids"].copy()
    return out

raw_dataset = Dataset.from_dict({"text": samples})
tokenized_dataset = raw_dataset.map(tokenize, remove_columns=["text"])
tokenized_dataset = tokenized_dataset.train_test_split(test_size=0.05, seed=42)
print(tokenized_dataset)

## 4. Load Pretrained InternVL2-4B Model (4-bit QLoRA)

In [ ]:
import torch
from transformers import AutoModelForCausalLM, BitsAndBytesConfig

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16,
)

model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    quantization_config=bnb_config,
    trust_remote_code=True,
    device_map="auto",
)
model.config.use_cache = False
print("Model loaded:", MODEL_ID)

## 5. Configure LoRA Fine-Tuning Parameters

In [ ]:
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training
from transformers import TrainingArguments

# Prepare model for k-bit training (enables gradient checkpointing + casts)
model = prepare_model_for_kbit_training(model)

lora_config = LoraConfig(
    r=16,                          # LoRA rank
    lora_alpha=32,                 # scaling factor
    target_modules=[               # InternVL2 attention projection layers
        "q_proj", "k_proj", "v_proj", "o_proj",
        "gate_proj", "up_proj", "down_proj",
    ],
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM",
)

model = get_peft_model(model, lora_config)
model.print_trainable_parameters()

OUTPUT_DIR = "/content/drive/MyDrive/cantopop/internvl2-4b-cantopop-lora"

training_args = TrainingArguments(
    output_dir=OUTPUT_DIR,
    num_train_epochs=5,
    per_device_train_batch_size=2,
    per_device_eval_batch_size=2,
    gradient_accumulation_steps=8,   # effective batch = 16
    learning_rate=2e-4,
    lr_scheduler_type="cosine",
    warmup_ratio=0.05,
    fp16=True,
    logging_steps=10,
    eval_strategy="epoch",
    save_strategy="epoch",
    save_total_limit=2,
    load_best_model_at_end=True,
    report_to="none",
    dataloader_pin_memory=False,
)
print("Training args configured.")

## 6. Define Training Loop & Start Fine-Tuning

In [ ]:
from transformers import Trainer, DataCollatorForLanguageModeling

data_collator = DataCollatorForLanguageModeling(tokenizer=tokenizer, mlm=False)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_dataset["train"],
    eval_dataset=tokenized_dataset["test"],
    data_collator=data_collator,
)

print(f"Training on {len(tokenized_dataset['train'])} samples, "
      f"evaluating on {len(tokenized_dataset['test'])} samples.")
trainer.train()

## 7. Save & Export Fine-Tuned Model

The LoRA adapter weights will be saved to Google Drive. To use the model later, load the base `InternVL2-4B` and merge the adapter.

In [ ]:
# Save LoRA adapter + tokenizer to Google Drive
model.save_pretrained(OUTPUT_DIR)
tokenizer.save_pretrained(OUTPUT_DIR)
print(f"LoRA adapter saved to: {OUTPUT_DIR}")

# ── Optional: merge adapter into base model and save full weights ──────────
# from peft import PeftModel
# base = AutoModelForCausalLM.from_pretrained(MODEL_ID, trust_remote_code=True, torch_dtype=torch.bfloat16)
# merged = PeftModel.from_pretrained(base, OUTPUT_DIR)
# merged = merged.merge_and_unload()
# merged.save_pretrained(OUTPUT_DIR + "-merged")
# tokenizer.save_pretrained(OUTPUT_DIR + "-merged")
# print("Merged model saved.")

# ── Optional: push adapter to Hugging Face Hub ─────────────────────────────
# from huggingface_hub import notebook_login
# notebook_login()
# model.push_to_hub("your-hf-username/internvl2-4b-cantopop-lora")
# tokenizer.push_to_hub("your-hf-username/internvl2-4b-cantopop-lora")